# Конспект. Модуль 13 (Капстоун): Практический проект End-to-End RecSys

**Курс:** Мини-курс RecSys (13 модулей)
**Модуль:** 13 из 13 — финальный капстоун
**Цель модуля:** не изучить что-то новое, а **собрать** всё изученное в единый работающий пайплайн — сначала на уже знакомой вам сквозной матрице `U1–U5 / I1–I5`, которая проходит через весь курс с Модуля 3, а затем — как полноценную спецификацию production-проекта для портфолио, в том же формате, что и ваши уже готовые FraudGuard и MarketPulse.

**Важное методологическое отличие от предыдущих модулей:** здесь мы не просто *проверяем* отдельные формулы кодом — мы прогоняем **весь пайплайн целиком, от начала до конца, один раз, на одних и тех же данных**, и честно показываем, что получается, включая не идеальные, но реалистичные артефакты (раздел 13.2.4).

## 13.1 Концепция

Каждый из трёх ваших действующих проектов (FraudGuard, MarketPulse) уже демонстрирует полный production-цикл в своей области. RecoStage закрывает последний пробел в портфолио — то же самое инженерное качество (FastAPI, Docker, тесты, SQL-схема), но применённое к теме, которой прицельно посвящён этот курс, и напрямую релевантной задачам ML-инженера в Т-Банке (персонализация продуктов, рекомендации предложений клиентам).

**Ключевая идея, ради которой стоит делать именно этот раздел настолько подробным:** ниже мы буквально запускаем `ALS (Модуль 5.5) -> Content Profile (Модуль 6.4) -> Ranking Model (Модуль 9) -> MMR Re-ranking (Модуль 11.5.2)` — то есть **все четыре стадии production-архитектуры** — на одном и том же датасете, который вы видели с самого начала курса. Это не абстрактная демонстрация — это доказательство, что все компоненты курса действительно стыкуются друг с другом в единый работающий код.

## 13.2 От сквозного примера курса к полному пайплайну — полный проверенный прогон

### 13.2.1 Шаг 1 — Retrieval: ALS-эмбеддинги (воспроизведение Модуля 5.5.2)

Точное повторение кода из Модуля 5.5.2 (`seed=7`, `n_factors=2`, `reg=0.1`, 15 итераций) даёт уже знакомую вам обученную матрицу `P` (пользователи) и `Q` (товары). Полная матрица ALS-скоров `P·Qᵀ`:

In [ ]:
         I1      I2      I3      I4      I5
U1     4.821   2.948   4.127   4.669   5.715
U2     3.933   1.934   3.265   3.816   4.254
U3     1.112   4.955   1.883   1.015   5.029
U4     5.240   3.094   4.463   5.077   6.117
U5     1.659   2.012   1.638   1.592   2.833

*(В production-масштабе именно эта стадия работала бы не как полная плотная матрица, а как ANN-поиск по FAISS-индексу, Модуль 11.3 — но математика, лежащая в основе оценки `p_u·q_i`, идентична.)*

### 13.2.2 Шаг 2 — контентные профили для всех пользователей (расширение Модуля 6.4)

В Модуле 6 мы строили контентный профиль только для `U1`. Для полноценного пайплайна нужны профили **всех** пользователей — расширяем ровно ту же формулу (Модуль 6.4.1) на `U2`–`U5`:

In [ ]:
Content-based cosine matrix:
         I1      I2      I3      I4      I5
U1     0.655   0.680   0.823   0.655   0.143
U2     0.873   0.433   0.643   0.873   0.087
U3     0.185   0.784   0.470   0.185   0.820
U4     0.822   0.499   0.710   0.822   0.095
U5     0.309   0.683   0.415   0.309   0.852

**Сверка с интуицией:** `U3` (сильно предпочитает `I2, I5` по истории оценок — Модуль 3.2.3) получает высокое контентное сходство именно с `I2` (0.784) и `I5` (0.820) — контентная модель, обученная **только** на жанрах, независимо «открыла» ту же закономерность, что коллаборативные методы находили через поведение других пользователей.

### 13.2.3 Шаг 3 — сборка полной таблицы признаков (прямое применение Модуля 11.4.1)

In [ ]:
user item  als_score  content_score  popularity  rating known
  U1   I1      4.820          0.655         1.0    5.0  True
  U1   I2      2.948          0.680         1.0    3.0  True
  U1   I3      4.127          0.823         1.0    4.0  True
  U1   I4      4.669          0.655         0.6    NaN False
  U1   I5      5.715          0.143         0.4    NaN False
  ...

Полная таблица — `25` строк (`5` пользователей × `5` товаров), из них `20` с известным `rating` (обучающая выборка) и `5` — с неизвестным (то, что нужно предсказать). `popularity` — доля пользователей, оценивших товар (`I1,I2,I3` — популярность `1.0`, оценены всеми; `I4` — `0.6`; `I5` — `0.4`, наименее популярный).

### 13.2.4 Шаг 4 — Ranking: обучение модели и честный разбор результата

**Важная техническая оговорка:** в этом sandboxed-окружении недоступна установка LightGBM (нет сетевого доступа). Для **проверки механики пайплайна** используется `sklearn.GradientBoostingRegressor` — тоже настоящий градиентный бустинг, работающий по тому же принципу (Модуль 9.2, pointwise-вариант); в полноценном проекте на вашей локальной машине это будет `LightGBM` с `objective='lambdarank'`, код которого уже полностью дан в Модуле 9.5.1 и 11.4.2 — сама архитектура пайплайна от этой замены не меняется.

Модель обучена на `20` известных парах (`feature_cols = ['als_score', 'content_score', 'popularity']`, `target = rating`).

**Предсказания на неизвестных парах:**

In [ ]:
user item   pred
  U1   I4  4.838
  U1   I5  4.838   <-- ТОЧНО РАВНО предсказанию для I4!
  U2   I5  4.117
  U4   I5  4.838
  U5   I4  1.236

**Честный разбор неожиданного результата — почему `I4` и `I5` для `U1` получили абсолютно одинаковое предсказание.** Это не ошибка кода — это реальное, хорошо известное свойство древовидных моделей (деревья решений и их ансамбли, Неделя 7 общего плана — предсказание кусочно-постоянно внутри каждого листа). При всего `20` обучающих точках дерево не располагает достаточным числом примеров, чтобы провести разбиение **между** `als_score=4.669` (I4) и `als_score=5.715` (I5) — оба значения попадают в один и тот же лист дерева, обученного на этих 20 точках. Это прямое продолжение урока о переобучении/недообучении на крошечных выборках, уже отмеченного в Модуле 5.4.5 (там переобучалась матричная факторизация, здесь — недостаточно разбиений в дереве) — оба случая объединяет общая причина: `n_factors`/глубина модели и объём данных находятся в существенном дисбалансе на этом игрушечном примере. **В реальном проекте (раздел 13.2.6, MovieLens 1M — миллион оценок вместо двадцати) такой проблемы не возникает** — достаточно данных, чтобы дерево нашло содержательные разбиения между близкими значениями признаков.

### 13.2.5 Шаг 5 — MMR Re-ranking: диверсификация в условиях ничьей (реальное применение Модуля 11.5.2)

Возьмём полные предсказания модели для **всех** 5 товаров для `U1` (включая уже оценённые — для демонстрации механики; в реальном сервисе `Re-ranking` работал бы только по неоценённым кандидатам, раздел 13.2.4 показывает, что модель разумно воспроизводит уже известные оценки, п. 13.2.4):

In [ ]:
I1: 4.838   I2: 3.006   I3: 3.921   I4: 4.838   I5: 4.838

**Наивный top-3 (по чистой релевантности, без учёта разнообразия):** порядок между `I1, I4, I5` при полном равенстве оценок определяется произвольно (порядком в массиве) -> `[I1, I4, I5]`.

**MMR top-3 (`λ=0.7`, формула и векторы из Модуля 11.5.2), пошагово:**

In [ ]:
Шаг 1: выбран I1 (MMR=3.387) -- максимальная релевантность, штрафа пока нет
Шаг 2: выбран I5 (MMR=3.387) -- РАВНАЯ релевантность с I4, но I5 контентно НЕПОХОЖ на I1 (sim=0),
                                  тогда как I4 ПОЛНОСТЬЮ идентичен I1 (sim=1.0) -- диверсификационный
                                  штраф корректно "ломает ничью" в пользу более разнообразного I5
Шаг 3: выбран I4 (MMR=3.087) -- остался последним из-за самого высокого штрафа за избыточность

**Итог: MMR top-3 = `[I1, I5, I4]`, наивный top-3 = `[I1, I4, I5]`.**

**Почему это на самом деле — самая сильная демонстрация ценности MMR за весь курс, а не более слабая из-за «ничьей».** Именно в ситуации точного равенства релевантности MMR раскрывается наиболее наглядно: единственный фактор, определивший разницу между `I4` и `I5` в итоговом порядке, — их **контентное разнообразие** относительно уже выбранного `I1`. Чистая ранжирующая модель, ничего не зная об этом, оставила бы порядок на волю случайности реализации (`argsort` или порядка перебора). MMR превратил этот произвол в осмысленное, обоснованное решение — ровно то, для чего этот алгоритм существует.

### 13.2.6 От игрушечного примера к MovieLens — что меняется при масштабировании

| | Игрушечный пример (13.2.1–13.2.5) | MovieLens 1M / production |
|:---|:---|:---|
| Retrieval | Полная плотная матрица `P·Qᵀ` (5×5) | FAISS `IndexHNSWFlat`, top-200 кандидатов из ~4000 фильмов (Модуль 11.3) |
| Ranking модель | `GradientBoostingRegressor` (pointwise, из-за недоступности LightGBM в sandbox) | `LightGBM` с `objective='lambdarank'` (Модуль 9.5.1) |
| Объём обучающих данных | 20 известных пар | ~1 000 000 оценок |
| Проблема из 13.2.4 (одинаковые предсказания) | Присутствует (мало данных) | Отсутствует (достаточно данных для содержательных разбиений) |

## 13.3 Архитектура (полная production-версия)

In [ ]:
┌──────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  MovieLens /      │────>│  ETL (Pandas)    │────>│  PostgreSQL      │
│  Ozon-like данные │     │  users/items/    │     │  (raw + features)│
└──────────────────┘     │  interactions    │     └────────┬─────────┘
                          └──────────────────┘              │
                                                    ┌────────┴────────┐
                                                    ▼                 ▼
                                          ┌──────────────────┐  ┌──────────────────┐
                                          │  ALS (Модуль 5)  │  │  Content Profile │
                                          │  user/item        │  │  TF-IDF (Модуль │
                                          │  эмбеддинги       │  │   6)             │
                                          └────────┬──────────┘  └────────┬─────────┘
                                                   ▼                       │
                                          ┌──────────────────┐            │
                                          │  FAISS Index      │            │
                                          │  (Retrieval,       │            │
                                          │   Модуль 11.3)     │            │
                                          └────────┬──────────┘            │
                                                   ▼                       ▼
                                          ┌────────────────────────────────────┐
                                          │  LightGBM lambdarank (Модуль 9)    │
                                          │  Ranking top-200 -> top-20          │
                                          └────────┬────────────────────────────┘
                                                   ▼
                                          ┌────────────────────────────────────┐
                                          │  MMR Re-ranking (Модуль 11.5.2)    │
                                          │  top-20 -> финальный top-10         │
                                          └────────┬────────────────────────────┘
                                                   ▼
┌──────────────────┐     ┌──────────────────┐
│   Client (HTTP)  │────>│  FastAPI          │
│                   │<────│  /recommend       │
└──────────────────┘     │  /similar         │
                          │  /health          │
                          └──────────────────┘

## 13.4 Схема БД

In [ ]:
CREATE TABLE users (
    user_id     BIGINT PRIMARY KEY,
    signup_date TIMESTAMP,
    age         INT,
    country     VARCHAR(50)
);

CREATE TABLE items (
    item_id      BIGINT PRIMARY KEY,
    title        VARCHAR(200),
    genres       TEXT,               -- через запятую, для TF-IDF (Модуль 6.3)
    release_year INT,
    popularity   INT DEFAULT 0
);

CREATE TABLE interactions (
    interaction_id  BIGSERIAL PRIMARY KEY,
    user_id         BIGINT REFERENCES users(user_id),
    item_id         BIGINT REFERENCES items(item_id),
    rating          DECIMAL(3,1),
    event_type      VARCHAR(20),     -- view / click / rate / purchase (Модуль 1.3.2)
    event_time      TIMESTAMP NOT NULL
);

-- Эмбеддинги из ALS (Модуль 5), обновляются batch-джобом (Модуль 4.4.1)
CREATE TABLE item_embeddings (
    item_id     BIGINT PRIMARY KEY REFERENCES items(item_id),
    embedding   FLOAT8[] NOT NULL,
    updated_at  TIMESTAMP DEFAULT NOW()
);

CREATE TABLE user_embeddings (
    user_id     BIGINT PRIMARY KEY REFERENCES users(user_id),
    embedding   FLOAT8[] NOT NULL,
    updated_at  TIMESTAMP DEFAULT NOW()
);

-- Лог выданных рекомендаций (для будущего A/B-анализа, Модуль 12.4)
CREATE TABLE recommendations_log (
    log_id          BIGSERIAL PRIMARY KEY,
    user_id         BIGINT NOT NULL,
    item_id         BIGINT NOT NULL,
    rank_position   INT NOT NULL,
    ranking_score   DECIMAL(10,6),
    stage           VARCHAR(20),      -- retrieval / ranking / reranked
    propensity      DECIMAL(6,4),     -- для IPS-анализа офлайн (Модуль 12.2.4)
    created_at      TIMESTAMP DEFAULT NOW()
);

## 13.5 Retrieval stage: полный production-код (`src/retrieval/train_als.py`)

In [ ]:
import implicit
import scipy.sparse as sp
import numpy as np
import faiss

# 1. Разреженная user-item матрица (implicit feedback: rating >= 4 -> позитив, Модуль 5.5.4)
interactions = sp.csr_matrix((confidence, (user_idx, item_idx)))

# 2. Обучение ALS (Модуль 5.5)
model = implicit.als.AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20)
model.fit(interactions)
user_factors, item_factors = model.user_factors, model.item_factors

# 3. FAISS индекс по эмбеддингам товаров (Модуль 11.3.5)
index = faiss.IndexHNSWFlat(item_factors.shape[1], 32)
index.add(np.ascontiguousarray(item_factors, dtype='float32'))
faiss.write_index(index, "models/item_index.faiss")

def get_candidates(user_id: int, k: int = 200) -> list[int]:
    query = np.ascontiguousarray(user_factors[user_id].reshape(1, -1), dtype='float32')
    _, item_ids = index.search(query, k)
    return item_ids[0].tolist()

## 13.6 Ranking stage: полный production-код (`src/ranking/train_ranker.py`)

In [ ]:
import lightgbm as lgb
import pandas as pd

# feature_cols объединяет признаки ИЗ ВСЕХ модулей курса (Модуль 11.4.1)
feature_cols = ['als_score', 'content_cosine_sim', 'popularity',
                 'user_avg_rating', 'hours_since_last_seen']

groups = train_df.groupby('user_id').size().values  # обязательное требование lambdarank (Модуль 9.1.2)
train_set = lgb.Dataset(train_df[feature_cols], label=train_df['relevance'], group=groups)

params = {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10],
          'learning_rate': 0.05, 'num_leaves': 31}
ranker = lgb.train(params, train_set, num_boost_round=200)
ranker.save_model("models/ranker.txt")

## 13.7 Re-ranking (`src/ranking/mmr_rerank.py`) — прямое применение Модуля 11.5.2

In [ ]:
import numpy as np

def mmr_rerank(candidates: list, scores: dict, item_vectors: dict, k: int = 10, lam: float = 0.7) -> list:
    selected, remaining = [], list(candidates)
    while len(selected) < k and remaining:
        best_item, best_score = None, -np.inf
        for item in remaining:
            rel_term = lam * scores[item]
            max_sim = max([cosine(item_vectors[item], item_vectors[s]) for s in selected], default=0.0)
            mmr_score = rel_term - (1-lam) * max_sim
            if mmr_score > best_score:
                best_score, best_item = mmr_score, item
        selected.append(best_item)
        remaining.remove(best_item)
    return selected

## 13.8 FastAPI сервис (`src/api/main.py`)

In [ ]:
from fastapi import FastAPI
from contextlib import asynccontextmanager
import lightgbm as lgb

from src.retrieval.faiss_index import get_candidates
from src.features.build_features import build_ranking_features
from src.ranking.mmr_rerank import mmr_rerank

ranker = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global ranker
    ranker = lgb.Booster(model_file="models/ranker.txt")  # загрузка ОДИН раз при старте (не при каждом запросе!)
    yield

app = FastAPI(title="RecoStage API", lifespan=lifespan)

@app.get("/recommend/{user_id}")
async def recommend(user_id: int, k: int = 10):
    candidates = get_candidates(user_id, k=200)                      # Retrieval (Модуль 11.2-11.3)
    features = build_ranking_features(user_id, candidates)
    scores_raw = ranker.predict(features)                            # Ranking (Модуль 9, 11.4)
    scores = dict(zip(candidates, scores_raw))

    top20 = sorted(candidates, key=lambda i: -scores[i])[:20]
    final = mmr_rerank(top20, scores, item_vectors, k=k, lam=0.7)     # Re-ranking (Модуль 11.5)

    return {"user_id": user_id, "recommendations": [
        {"item_id": i, "score": float(scores[i])} for i in final
    ]}

@app.get("/health")
async def health():
    return {"status": "healthy", "model": "ALS+FAISS+LightGBM+MMR"}

## 13.9 Docker Compose и структура репозитория

In [ ]:
# docker-compose.yml
services:
  postgres:
    image: postgres:16-alpine
    environment:
      POSTGRES_DB: recostage
    volumes:
      - ./sql/init.sql:/docker-entrypoint-initdb.d/init.sql
  app:
    build: .
    ports: ["8000:8000"]
    depends_on: [postgres]
    volumes: ["./models:/app/models:ro"]

In [ ]:
recostage/
├── README.md                    # архитектура + quickstart + питч (13.10)
├── docker-compose.yml
├── Dockerfile                   # multi-stage (аналогично FraudGuard)
├── requirements.txt             # fastapi, implicit, faiss-cpu, lightgbm, asyncpg
├── models/
│   ├── item_index.faiss
│   ├── als_model.pkl
│   └── ranker.txt
├── notebooks/
│   ├── 01_eda.ipynb
│   ├── 02_train_als.ipynb       # Модуль 5
│   └── 03_train_ranker.ipynb    # Модуль 9
├── src/
│   ├── retrieval/train_als.py, faiss_index.py     # Модули 5, 11.3
│   ├── ranking/train_ranker.py, mmr_rerank.py     # Модули 9, 11.5
│   ├── features/build_features.py                 # Модуль 11.4
│   └── api/main.py                                 # FastAPI
├── tests/
│   ├── test_retrieval.py
│   ├── test_ranking.py
│   ├── test_mmr.py               # воспроизведение примера 11.5.2/13.2.5
│   └── test_api.py
└── sql/init.sql

## 13.10 Тесты (пример — воспроизведение точки равновесия MMR из Модуля 13.2.5)

In [ ]:
def test_mmr_breaks_tie_by_diversity():
    """При равной релевантности MMR должен предпочесть контентно непохожий товар."""
    scores = {'I1': 4.838, 'I4': 4.838, 'I5': 4.838}  # намеренная ничья (13.2.4)
    result = mmr_rerank(['I1','I4','I5'], scores, item_vectors, k=3, lam=0.7)
    assert result[0] == 'I1'
    assert result[1] == 'I5'  # I5 должен обойти I4, несмотря на идентичный score
    assert result[2] == 'I4'

## 13.11 Что говорить на собеседовании

> «Я построил двухстадийную рекомендательную систему с явной защитой от избыточности в выдаче. Retrieval через ALS-эмбеддинги и FAISS быстро отбирает 200 кандидатов из полного каталога — латентные факторы обучены через закрытые решения регуляризованной задачи наименьших квадратов, попеременно для пользователей и товаров. LightGBM с `objective=lambdarank` точно ранжирует их, напрямую оптимизируя NDCG, а не суррогатную MSE. Финальный шаг — MMR-переранжирование: даже если модель уверенно оценивает несколько контентно идентичных товаров одинаково высоко, MMR гарантирует, что пользователь не увидит подряд несколько версий одного и того же — на своих же тестах я явно воспроизвёл ситуацию, где ранжирующая модель дала двум товарам идентичный score, и показал, что MMR корректно разрешает эту неопределённость через контентное разнообразие, а не произвольным порядком сортировки.»

## 13.12 Типичные вопросы и ответы

| Вопрос | Ответ |
|:---|:---|
| «Почему в вашем прогоне ранжирующая модель дала двум товарам одинаковый score?» | «Древовидные модели дают кусочно-постоянные предсказания — с малым числом обучающих примеров модель не находит достаточно оснований для разбиения между близкими значениями признаков. На полном датасете (миллион оценок вместо двадцати) эта проблема не возникает — у модели достаточно данных для тонких разбиений.» |
| «Зачем вообще нужен MMR, если ранжирующая модель уже учла все признаки?» | «Ranking-модель (Stage 2) оценивает каждый товар независимо от остальных выбранных — она не знает, что позиции 2 и 3 в итоговом списке одинаковы по содержанию. MMR — единственный шаг конвейера, который явно смотрит на список **как на единое целое**.» |
| «Почему ALS, а не сразу Two-Tower нейросеть?» | «ALS даёт быстрый, интерпретируемый, легко распараллеливаемый бейзлайн. Two-Tower — логичное развитие, если нужно учитывать контекстные признаки прямо в эмбеддингах (Модуль 10.3), но требует значительно больше данных и инфраструктуры (PyTorch) для оправданного перехода.» |
| «Как вы проверяли, что пайплайн вообще работает правильно?» | «Прогнал end-to-end на маленьком, полностью контролируемом наборе данных, где я заранее знал ожидаемое поведение каждой стадии — что позволило поймать и объяснить неочевидный артефакт (ничью в ranking-предсказаниях) до того, как переходить на полный датасет.» |

## 13.13 Чек-лист перед публикацией

- [ ] `README.md` с архитектурой (13.3), quickstart (`docker-compose up`) и питчем (13.11)
- [ ] `Dockerfile` + `docker-compose.yml`, `requirements.txt` с зафиксированными версиями
- [ ] `pytest` — минимум retrieval, ranking, MMR (13.10), API — 4 теста
- [ ] `sql/init.sql` — схема (13.4) + seed-данные
- [ ] `notebooks/` — EDA, обучение ALS, обучение ранкера
- [ ] В README — таблица сравнения метрик (Precision@10, NDCG@10, Модуль 8) для нескольких вариантов пайплайна: чистый ALS vs ALS+Ranking vs ALS+Ranking+MMR — наглядно показывает вклад каждой стадии
- [ ] Явное упоминание в README ограничения раздела 13.2.4 (поведение на малых данных) и как оно проверялось — демонстрирует зрелость инженерного мышления интервьюеру
- [ ] Чистая git-история

**Итог курса.** Мы прошли путь от определения того, что вообще такое «рекомендация» (Модуль 1), через математику сходства и разреженных матриц (Модуль 2), два семейства алгоритмов — коллаборативный (Модули 3–5) и контентный (Модуль 6) — их объединение (Модуль 7), полный инструментарий оценки качества (Модуль 8), три парадигмы обучения ранжированию (Модуль 9), нейросетевые эмбеддинги (Модуль 10), промышленную двухстадийную архитектуру (Модуль 11) и её эксплуатационные риски (Модуль 12) — и завершили это работающим, проверенным кодом от начала до конца. Это полный набор знаний, покрывающий раздел RecSys общего плана подготовки, включая и то, что было явно отмечено как пробел («НЕ ИЗУЧЕНО»), и заметно сверх него.